In [5]:
from selenium import webdriver 
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service # Service 모듈 추가
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import pyperclip
import time
import random 

print("=" *80)
print(" 개인프로젝트 네이버 블로그 자동 서로이웃 추가 프로그램 (우회 공격 + 동적 팝업 제어)")
print("=" *80)
print("\n")

# -------------------------------------------------------------
# 1. 사용자 입력 받기 (ID, PW, 목표 이웃 수)
# [주의] 주피터 노트북 환경에서는 이 input() 창이 셀 하단이나 화면 상단에 
# 조그맣게 뜨므로, 입력 전까지 커널이 무한 대기 상태([*])에 빠집니다.
# 만약 계속 멈춰있다면 아래 3줄을 지우고 직접 문자열로 ID/PW를 하드코딩하세요.
# -------------------------------------------------------------
v_id = input('🔑 네이버 로그인 ID를 입력하세요: ')
v_passwd = input('🔑 네이버 로그인 비밀번호를 입력하세요: ')
target_count = int(input('🎯 몇 명에게 서로이웃을 신청할까요? (숫자만 입력): '))

message_text = "서로이웃해요~블로그 자주 방문하고 좋아요 누르고 갑니다~"
current_count = 0 # 현재 추가한 이웃 수

print("\n🚀 서로이웃 추가 자동화를 시작합니다. 브라우저가 열리면 잠시 지켜봐주세요!")

options = Options()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--disable-blink-features=AutomationControlled")

# [요청 반영] 구버전 방식 유지: 지정하신 폴더 내의 chromedriver.exe 경로 지정
# 역슬래시(\) 대신 슬래시(/)를 사용하거나 r"경로" 형태로 써야 파이썬이 경로 오류를 내지 않습니다.
s = Service("C:/py_temp/chromedriver/chromedriver.exe")
driver = webdriver.Chrome(service=s, options=options) 

# 초기 URL 접근 (세션 워밍업 및 브라우저 히스토리 생성을 통한 봇 탐지 우회)
# 💡 전세계 최고 수준의 우회 기법 (Referer 일치화): 엉뚱한 사이트가 아닌 네이버 블로그 홈을 
# 먼저 밟아줌으로써, 이후 로그인 페이지로 넘어갈 때 네이버 서버가 100% 사람으로 신뢰하게 만듭니다.
base_url = 'https://section.blog.naver.com/'
driver.get(base_url)
time.sleep(random.uniform(2, 4)) # 기계적인 2초 대기 대신 사람처럼 2~4초 사이 랜덤 대기
driver.maximize_window()

wait = WebDriverWait(driver, 10)
actions = ActionChains(driver)

try:
    # -------------------------------------------------------------
    # 2. 우회 로그인 처리 (기존 코드 유지)
    # -------------------------------------------------------------
    print(">> 다이렉트 접근 및 우회 로그인 시도...")
    driver.get(f"https://blog.naver.com/{v_id}?Redirect=Write")
    time.sleep(3)

    # 아이디 입력
    id_element = driver.find_element(By.NAME, 'id')
    id_element.click()
    pyperclip.copy(v_id) 
    actions.key_down(Keys.CONTROL).send_keys('v').key_up(Keys.CONTROL).perform()
    time.sleep(1)

    # 비밀번호 입력
    pw_element = driver.find_element(By.NAME, 'pw')
    pw_element.click()
    pyperclip.copy(v_passwd) 
    actions.key_down(Keys.CONTROL).send_keys('v').key_up(Keys.CONTROL).perform()
    time.sleep(1)

    # 로그인 버튼 클릭
    driver.find_element(By.ID, 'log.login').click()  
    print(">> 로그인 완료. 메인 페이지로 이동합니다.")
    time.sleep(5)  
    
    # -------------------------------------------------------------
    # 3. 네이버 메인 -> 블로그 홈 -> 주제별 보기(세계여행) 이동
    # -------------------------------------------------------------
    driver.get("https://www.naver.com/")
    time.sleep(2)
    
    # 메인에서 '블로그' 탭 클릭
    driver.find_element(By.XPATH, "//a[contains(@class, 'MyView-module__item_link') and .//span[text()='블로그']]").click()
    time.sleep(2)
    
    # 블로그 홈버튼 누르기 (새 창으로 열림)
    driver.find_element(By.XPATH, "//a[contains(@class, 'MyView-module__link_service') and contains(text(), '블로그')]").click()
    time.sleep(random.uniform(3, 5)) # 랜덤 슬립
    
    # 새 창으로 제어권 넘기기
    driver.switch_to.window(driver.window_handles[-1])
    main_blog_window = driver.current_window_handle # 현재 블로그 리스트 창 기억
    
    # '주제별 보기' 클릭
    wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(text(), '주제별 보기')]"))).click()
    time.sleep(2)
    
    # '세계여행' 카테고리 클릭
    wait.until(EC.element_to_be_clickable((By.XPATH, "//a[.//span[text()='국내여행']]"))).click()
    time.sleep(3)
    
    print(">> [국내여행] 카테고리 진입 완료. 탐색을 시작합니다.")

    # -------------------------------------------------------------
    # 4. 반복문: 게시글 리스트 돌면서 서로이웃 신청
    # -------------------------------------------------------------
    while current_count < target_count:
        # 현재 페이지의 10개 게시물 링크 추출 (Stale Element Reference 방지를 위해 href만 먼저 뽑아냄)
        post_elements = driver.find_elements(By.CSS_SELECTOR, "a.desc_inner")
        post_urls = [elem.get_attribute("href") for elem in post_elements]
        
        for url in post_urls:
            if current_count >= target_count:
                break
                
            # 새 탭 열어서 게시글 접속 (기존 리스트 페이지 유지)
            driver.execute_script(f"window.open('{url}', '_blank');")
            driver.switch_to.window(driver.window_handles[-1])
            time.sleep(random.uniform(2, 4))
            
            try:
                # [중요] 블로그 포스팅 본문과 이웃추가 버튼은 mainFrame 안에 있음
                driver.switch_to.frame("mainFrame")
                
                # 이웃추가 버튼 찾기
                try:
                    add_buddy_btn = driver.find_element(By.CSS_SELECTOR, "a.btn_buddy")
                except:
                    # 이웃추가 버튼이 없으면 패스
                    print("   [-] 이웃추가 버튼이 없습니다. 패스합니다.")
                    driver.close()
                    driver.switch_to.window(main_blog_window)
                    continue

                # 버튼이 있으면 클릭
                add_buddy_btn.click()
                time.sleep(2)
                
                # 팝업 창으로 제어권 넘기기 (이웃추가를 누르면 팝업창이 뜸)
                windows = driver.window_handles
                driver.switch_to.window(windows[-1])
                
                # --- [경우의 수 1] 이미 서로이웃인 경우 ---
                if len(driver.find_elements(By.XPATH, "//*[contains(text(), '님과 현재 서로이웃입니다')]")) > 0:
                    print("   [-] 이미 서로이웃입니다. 패스합니다.")
                    driver.close() # 팝업 닫기
                    driver.switch_to.window(windows[-2]) # 포스팅 창으로
                    driver.close() # 포스팅 닫기
                    driver.switch_to.window(main_blog_window) # 메인으로
                    continue
                
                # --- [경우의 수 2] 서로이웃을 받지 않는 경우 ---
                if len(driver.find_elements(By.XPATH, "//*[contains(text(), '서로이웃 신청을 받지 않는 이웃입니다')]")) > 0:
                    print("   [-] 서로이웃 신청을 받지 않는 블로거입니다. 패스합니다.")
                    driver.close() 
                    driver.switch_to.window(windows[-2]) 
                    driver.close() 
                    driver.switch_to.window(main_blog_window) 
                    continue

                # --- [경우의 수 3] 서로이웃 신청 진행 ---
                # '서로이웃' 라디오 버튼 클릭
                try:
                    both_buddy_label = wait.until(EC.element_to_be_clickable((By.XPATH, "//label[@for='each_buddy_add']")))
                    both_buddy_label.click()
                except:
                    # 클릭 불가시 (라디오 버튼 비활성화 등) 예외처리
                    driver.close() 
                    driver.switch_to.window(windows[-2]) 
                    driver.close() 
                    driver.switch_to.window(main_blog_window) 
                    continue
                
                time.sleep(1)
                
                # 첫번째 '다음' 버튼 클릭
                driver.find_element(By.CSS_SELECTOR, "a._buddyAddNext").click()
                time.sleep(1.5)
                
                # --- [경우의 수 4] 메시지 입력창 확인 및 전송 ---
                try:
                    textarea = driver.find_element(By.ID, "message")
                    textarea.clear()
                    
                    # 봇 탐지 우회를 위해 사람처럼 한글자씩 타이핑
                    for char in message_text:
                        textarea.send_keys(char)
                        time.sleep(random.uniform(0.01, 0.05))
                    
                    time.sleep(1)
                    
                    # 두번째 '다음' 버튼 클릭 (최종 신청)
                    driver.find_element(By.CSS_SELECTOR, "a._addBothBuddy").click()
                    time.sleep(1.5)
                    
                    # '닫기' 버튼 클릭
                    driver.find_element(By.CSS_SELECTOR, "a.button_close").click()
                    
                    current_count += 1
                    print(f"   [+] 서로이웃 신청 완료! (현재 진행: {current_count}/{target_count})")
                    
                except:
                    print("   [-] 메시지 창을 찾을 수 없거나 에러 발생. 패스합니다.")
                    driver.close() # 팝업 닫기
                
                # 열려있던 포스팅 창 닫고 메인 리스트로 복귀
                driver.switch_to.window(windows[-2])
                driver.close()
                driver.switch_to.window(main_blog_window)
                time.sleep(random.uniform(2, 4)) # 매크로 탐지 우회를 위한 휴식
                
            except Exception as e:
                # 중간에 알수없는 에러 발생시 창들 정리 후 메인으로 복귀
                for handle in driver.window_handles:
                    if handle != main_blog_window:
                        driver.switch_to.window(handle)
                        driver.close()
                driver.switch_to.window(main_blog_window)
                print("   [!] 포스팅 처리 중 에러 발생, 다음 글로 넘어갑니다.", e)
                continue
                
        # 10개(한 페이지)를 다 돌았는데 아직 목표치에 도달하지 못했다면 페이징 처리
        if current_count < target_count:
            print(">> 한 페이지를 모두 탐색했습니다. 다음 페이지로 이동합니다.")
            
            # 현재 선택된 페이지 번호 찾기 (aria-current="page" 활용)
            try:
                current_page_elem = driver.find_element(By.CSS_SELECTOR, "a[aria-current='page']")
                current_page_num = int(current_page_elem.text)
                next_page_num = current_page_num + 1
                
                try:
                    # 다음 숫자 페이지 버튼 찾아서 클릭
                    next_page_btn = driver.find_element(By.CSS_SELECTOR, f"a[aria-label='{next_page_num}페이지']")
                    next_page_btn.click()
                except:
                    # 10페이지 단위가 넘어가서 숫자가 안보일 경우 '다음' 화살표 그룹 버튼 클릭
                    next_group_btn = driver.find_element(By.CSS_SELECTOR, "a.button_next")
                    next_group_btn.click()
                
                time.sleep(4) # 페이지 로딩 대기
            except Exception as page_e:
                print(">> 페이징 처리 중 마지막 페이지에 도달했거나 에러가 발생했습니다. 프로그램을 종료합니다.", page_e)
                break

    print("\n🎉 목표한 서로이웃 신청 횟수를 모두 채웠습니다! 프로그램을 종료합니다.")
    time.sleep(5)

except Exception as e:
    print("\n[치명적 에러 발생] 프로그램 실행 도중 문제가 발생했습니다:", e)

finally:
    driver.quit()

 개인프로젝트 네이버 블로그 자동 서로이웃 추가 프로그램 (우회 공격 + 동적 팝업 제어)



🚀 서로이웃 추가 자동화를 시작합니다. 브라우저가 열리면 잠시 지켜봐주세요!
>> 다이렉트 접근 및 우회 로그인 시도...
>> 로그인 완료. 메인 페이지로 이동합니다.

[치명적 에러 발생] 프로그램 실행 도중 문제가 발생했습니다: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=146.0.7680.165); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff7734eaa55
	0x7ff773248630
	0x7ff772fdd75d
	0x7ff772fc8c72
	0x7ff772fef015
	0x7ff7730690e0
	0x7ff773085282
	0x7ff773029098
	0x7ff773029f83
	0x7ff773517810
	0x7ff773511afd
	0x7ff773532c1a
	0x7ff773263345
	0x7ff77326b81c
	0x7ff773251924
	0x7ff773251ad6
	0x7ff773237e47
	0x7ffb5d36e8d7
	0x7ffb5ebac48c

